<a href="https://colab.research.google.com/github/hypher777/tiny-gpt-from-scratch/blob/main/tiny_gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import sys

print("Python version:", sys.version)

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("We will start with CPU.")
except ImportError:
    print("PyTorch is not installed yet.")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cpu
CUDA available: False
We will start with CPU.


In [ ]:
text = """
The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.
"""

print("training text :")
print(text)
print("number of text :")
print(len(text))
characters=sorted(set(text))
print("\n unique characters :")
print(characters)
print("\n number of unique characters :")
print(len(characters))

training text :

The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.

number of text :
449

 unique characters :
['\n', ' ', '.', 'A', 'E', 'N', 'P', 'S', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']

 number of unique characters :
31


In [ ]:

# Our tiny training text
text = """
The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.
"""


# Collect every unique character and sort them
characters = sorted(set(text))

# Give each character a unique number
stoi = {}
for index, character in enumerate(characters):
    stoi[character] = index

# Convert numbers back into characters
itos = {}
for character, index in stoi.items():
    itos[index] = character


# Convert text into a list of numbers
def encode(sentence):
    token_ids = []

    for character in sentence:
        token_ids.append(stoi[character])

    return token_ids


# Convert a list of numbers back into text
def decode(token_ids):
    sentence = ""

    for index in token_ids:
        sentence += itos[index]

    return sentence


# Test both directions
encoded = encode("hello")
decoded = decode(encoded)

print("Encoded:", encoded)
print("Decoded:", decoded)
print("Correct:", decoded == "hello")


Encoded: [16, 13, 19, 19, 22]
Decoded: hello
Correct: True


In [ ]:

import torch

# Convert every character into its number.
# PyTorch tensors are the data format our model will use.
data = torch.tensor(encode(text), dtype=torch.long)

# Check how many character tokens we have.
print("Total tokens:", len(data))


Total tokens: 449


In [ ]:

# How many characters the model can look at at once.
block_size = 8

# Input: the first 8 characters.
x = data[:block_size]

# Target: the same sequence shifted forward by one character.
y = data[1:block_size + 1]

# Decode the numbers so we can understand the example.
print("Input: ", repr(decode(x.tolist())))
print("Target:", repr(decode(y.tolist())))


Input:  '\nThe str'
Target: 'The stro'


In [ ]:

# How many training examples we process together
batch_size = 4

# Number of characters the model can look at
# This should already be 8 from our previous module.
block_size = 8

# Pick random starting positions in the training text.
# Leave enough room for the input and its next-character targets.
starts = torch.randint(
    len(data) - block_size,
    (batch_size,)
)

# Store the input sequences and their targets.
x_batch = []
y_batch = []

for start in starts:
    # Convert the random position into a normal Python integer.
    start = start.item()

    # Input: a sequence of characters from this position.
    x = data[start : start + block_size]

    # Target: the same sequence shifted forward by one character.
    y = data[start + 1 : start + block_size + 1]

    x_batch.append(x)
    y_batch.append(y)

# Combine the individual sequences into two tensors.
x_batch = torch.stack(x_batch)
y_batch = torch.stack(y_batch)

print("Input batch shape:", x_batch.shape)
print("Target batch shape:", y_batch.shape)


Input batch shape: torch.Size([4, 8])
Target batch shape: torch.Size([4, 8])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Number of unique characters in our training text
vocab_size = len(characters)

# A tiny model: each input character gets scores for every possible next character
model = nn.Embedding(vocab_size, vocab_size)

print("Vocabulary size:", vocab_size)
print(model)



Vocabulary size: 31
Embedding(31, 31)


In [ ]:
# Give the model the first character in our training text
input_character = data[0]

# Calculate scores for every possible next character
logits = model(input_character)

print("Input character:", decode([input_character.item()]))
print("Scores shape:", logits.shape)


Input character: 

Scores shape: torch.Size([31])


In [ ]:
# Every character except the last is an input
inputs = data[:-1]

# Every character except the first is its corresponding target
targets = data[1:]

# Predict the next character for every input
logits = model(inputs)

# Measure how far the predictions are from the correct targets
loss = F.cross_entropy(logits, targets)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([448])
Target shape: torch.Size([448])
Initial loss: 3.8715460300445557


In [ ]:

# Create an optimizer connected to our current model.
# If you recreate the model, you must recreate the optimizer too.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Optimizer ready!")

Optimizer ready!


In [68]:
# Train the model repeatedly
for step in range(100):

    # 1. Clear gradients from the previous step
    optimizer.zero_grad()

    # 2. Predict the next character
    logits = model(inputs)

    # 3. Measure prediction error
    loss = F.cross_entropy(logits, targets)

    # 4. Calculate gradients
    loss.backward()

    # 5. Update the model's parameters
    optimizer.step()

    # Print progress every 20 steps
    if (step + 1) % 20 == 0:
        print(f"Step {step + 1}: Loss = {loss.item():.4f}")

Step 20: Loss = 3.5878
Step 40: Loss = 3.3365
Step 60: Loss = 3.1199
Step 80: Loss = 2.9351
Step 100: Loss = 2.7780


In [69]:
# Start generation with a single character
current = torch.tensor([stoi["T"]], dtype=torch.long)

# Generate 100 characters
generated = []

with torch.no_grad():
    for _ in range(100):
        # Predict the next character
        logits = model(current[-1])

        # Convert scores into probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Choose the next character
        next_character = torch.multinomial(probabilities, num_samples=1)

        # Save it and use it for the next prediction
        generated.append(next_character.item())
        current = torch.cat([current, next_character])

print(decode(generated))

t cA atNAvote ypNnskyTwepnevSalciNmbso
y cnr TthTdeSweanEoapngnwwlwer pfemiglTth y t yTronsk.SaciA v
